In [12]:
"""
================================================================================
Synthetic Health Data Generation Pipeline (SDV: Gaussian Copula, CTGAN, TVAE)
================================================================================

This module provides a reusable, dataset-agnostic pipeline to generate
synthetic tabular datasets with three SDV single-table synthesizers:

    - GaussianCopulaSynthesizer  (fit once, sample many times)
    - CTGANSynthesizer           (retrained at every run)
    - TVAESynthesizer            (retrained at every run)

The pipeline was originally developed for the PIMA diabetes dataset and is
generalised here so that the exact same procedure can be applied to other
datasets (e.g. AIDS, REIN) by simply providing a per-dataset configuration
dictionary. Hyperparameters of the generators are expected to differ across
datasets, so they are supplied explicitly per dataset rather than hard-coded.

For each dataset and each synthesizer, the pipeline:
    1. Loads the training data and detects SDV metadata.
    2. Runs `n_runs` independent generations (fixing seeds for reproducibility).
    3. Saves every synthetic dataset to disk.
    4. Records timing, synthesizer parameters, and a hash of each generated
       dataset (to check for duplicated/degenerate outputs).
    5. Saves a run summary, a parameter summary, and an environment snapshot
       as CSV files, and prints a short report.

Usage
-----
Run the whole study (all datasets, all synthesizers) with a single call:

    >>> run_all_datasets(DATASET_CONFIGS)

Run a single dataset:

    >>> run_dataset_pipeline("PIMA", **DATASET_CONFIGS["PIMA"])

Edit `DATASET_CONFIGS` at the bottom of this file to point to your own data
paths, output directories, and per-dataset generator hyperparameters.
"""

# ==============================================================================
# Imports
# ==============================================================================

import os
import sys
import time
import random
import platform
from typing import Any, Callable, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch

import sdv
from sdv.metadata import Metadata
from sdv.single_table import (
    CTGANSynthesizer,
    GaussianCopulaSynthesizer,
    TVAESynthesizer,
)


# ==============================================================================
# Reproducibility
# ==============================================================================

def set_seed(seed: int = 0, verbose: bool = True) -> None:
    """
    Enable deterministic mode and set all relevant random seeds
    (Python, NumPy, PyTorch, CUDA).

    Parameters
    ----------
    seed : int
        Seed value applied to every random number generator.
    verbose : bool
        If True, print a short confirmation message.
    """
    if verbose:
        print(f"DETERMINISTIC MODE enabled (seed={seed})")

    torch.manual_seed(seed)
    random.seed(seed)
    np.random.seed(seed)

    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

    torch.use_deterministic_algorithms(True)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def generate_seeds(n_runs: int, base_seed: int = 0) -> List[int]:
    """
    Deterministically generate `n_runs` independent seeds from a master seed.

    Parameters
    ----------
    n_runs : int
        Number of seeds to generate (one per experimental run).
    base_seed : int
        Master seed controlling the whole sequence of seeds.

    Returns
    -------
    List[int]
        List of `n_runs` integer seeds.
    """
    rng = np.random.default_rng(base_seed)
    return rng.integers(low=0, high=2**31 - 1, size=n_runs).tolist()


# ==============================================================================
# Environment logging
# ==============================================================================

def get_environment_info() -> pd.DataFrame:
    """
    Collect software/hardware environment information for reproducibility
    and publication purposes.

    Returns
    -------
    pd.DataFrame
        Single-row DataFrame describing the execution environment.
    """
    info: Dict[str, Any] = {
        "python_version": sys.version,
        "platform": platform.platform(),
        "sdv_version": sdv.__version__,
        "pandas_version": pd.__version__,
        "numpy_version": np.__version__,
        "torch_version": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "cuda_version": torch.version.cuda,
    }
    info["gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    return pd.DataFrame([info])


# ==============================================================================
# Data loading and metadata detection
# ==============================================================================

def load_dataset(data_path: str, table_name: str) -> Tuple[pd.DataFrame, Metadata]:
    """
    Load a CSV dataset and automatically detect its SDV metadata.

    Parameters
    ----------
    data_path : str
        Path to the training CSV file.
    table_name : str
        Logical name of the table, used by SDV metadata.

    Returns
    -------
    (pd.DataFrame, Metadata)
        The loaded dataframe and its detected SDV metadata.
    """
    df = pd.read_csv(data_path,na_values=["NA"], keep_default_na=False)

    print("\nDataset information")
    print("-------------------")
    print(f"Dataset name : {table_name}")
    print(f"Rows         : {df.shape[0]}")
    print(f"Columns      : {df.shape[1]}")

    metadata = Metadata.detect_from_dataframe(data=df, table_name=table_name.lower())
    return df, metadata


# ==============================================================================
# Synthesizer trainers
# ==============================================================================

def train_gaussian_copula(
    df: pd.DataFrame,
    metadata: Metadata,
    seed: int,
    enforce_min_max_values: bool = True,
    enforce_rounding: bool = True,
    **_ignored: Any,
) -> GaussianCopulaSynthesizer:
    """
    Fit a GaussianCopulaSynthesizer. Fitting is deterministic; only the
    subsequent sampling step benefits from seeding.
    """
    set_seed(seed)
    synthesizer = GaussianCopulaSynthesizer(
        metadata=metadata,
        enforce_min_max_values=enforce_min_max_values,
        enforce_rounding=enforce_rounding,
    )
    synthesizer.fit(df)
    return synthesizer


def train_ctgan(
    df: pd.DataFrame,
    metadata: Metadata,
    seed: int,
    epochs: int,
    batch_size: int,
    generator_dim: Tuple[int, ...],
    discriminator_dim: Tuple[int, ...],
    cuda: bool = False,
    verbose: bool = True,
    **_ignored: Any,
) -> CTGANSynthesizer:
    """
    Train a CTGANSynthesizer with a fixed random seed set beforehand.
    """
    set_seed(seed)
    synthesizer = CTGANSynthesizer(
        metadata=metadata,
        epochs=epochs,
        batch_size=batch_size,
        generator_dim=generator_dim,
        discriminator_dim=discriminator_dim,
        verbose=verbose,
        cuda=cuda,
    )
    synthesizer.fit(df)
    return synthesizer


def train_tvae(
    df: pd.DataFrame,
    metadata: Metadata,
    seed: int,
    epochs: int,
    batch_size: int,
    embedding_dim: int,
    compress_dims: Tuple[int, ...],
    decompress_dims: Tuple[int, ...],
    l2scale: float,
    cuda: bool = False,
    verbose: bool = True,
    **_ignored: Any,
) -> TVAESynthesizer:
    """
    Train a TVAESynthesizer with a fixed random seed set beforehand.
    """
    set_seed(seed)
    synthesizer = TVAESynthesizer(
        metadata=metadata,
        epochs=epochs,
        batch_size=batch_size,
        embedding_dim=embedding_dim,
        compress_dims=compress_dims,
        decompress_dims=decompress_dims,
        l2scale=l2scale,
        cuda=cuda,
        verbose=verbose,
    )
    synthesizer.fit(df)
    return synthesizer


# Registry mapping a synthesizer name to its training function and to whether
# it must be retrained at every run (CTGAN/TVAE) or fitted once and only
# re-sampled (GaussianCopula).
SYNTHESIZER_TRAINERS: Dict[str, Callable[..., Any]] = {
    "GaussianCopula": train_gaussian_copula,
    "CTGAN": train_ctgan,
    "TVAE": train_tvae,
}

RETRAIN_EACH_RUN: Dict[str, bool] = {
    "GaussianCopula": False,  # fit once, sample n_runs times
    "CTGAN": True,            # retrain at every run
    "TVAE": True,             # retrain at every run
}


# ==============================================================================
# Core experiment runner (generic across synthesizers and datasets)
# ==============================================================================

def _hash_dataframe(df: pd.DataFrame) -> int:
    """Return a hash of a dataframe's content, used to detect duplicated runs."""
    return int(pd.util.hash_pandas_object(df, index=True).sum())


def run_synthesizer_experiment(
    df: pd.DataFrame,
    metadata: Metadata,
    dataset_name: str,
    synth_name: str,
    synth_params: Dict[str, Any],
    seeds: List[int],
    output_dir: str,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Run `len(seeds)` independent synthetic-data generations for a given
    dataset/synthesizer combination and save every generated dataset to disk.

    Handles both training regimes transparently:
        - Synthesizers that must be retrained at every run (CTGAN, TVAE).
        - Synthesizers fitted once and re-sampled multiple times
          (GaussianCopula), where seeding only affects the sampling step.

    Parameters
    ----------
    df : pd.DataFrame
        Training data.
    metadata : Metadata
        SDV metadata describing `df`.
    dataset_name : str
        Short dataset identifier used in filenames (e.g. "PIMA", "AIDS", "REIN").
    synth_name : str
        Synthesizer identifier. Must be a key of `SYNTHESIZER_TRAINERS`.
    synth_params : dict
        Hyperparameters forwarded to the corresponding training function.
    seeds : list of int
        One seed per run.
    output_dir : str
        Directory where synthetic datasets and result tables are saved.

    Returns
    -------
    (run_summary_df, parameters_df)
        DataFrame of per-run timings/hashes, and DataFrame of synthesizer
        parameters used at each run.
    """
    os.makedirs(output_dir, exist_ok=True)

    train_fn = SYNTHESIZER_TRAINERS[synth_name]
    retrain_each_run = RETRAIN_EACH_RUN[synth_name]
    n_runs = len(seeds)

    run_results: List[Dict[str, Any]] = []
    parameter_results: List[Dict[str, Any]] = []

    fitted_synthesizer = None
    base_training_time: Optional[float] = None
    base_parameters: Optional[Dict[str, Any]] = None

    # For synthesizers fitted only once, do it here, ahead of the loop.
    if not retrain_each_run:
        start_time = time.time()
        fitted_synthesizer = train_fn(df, metadata, seed=seeds[0], **synth_params)
        base_training_time = time.time() - start_time
        base_parameters = fitted_synthesizer.get_parameters()

    experiment_start_time = time.time()

    for run_id, seed in enumerate(seeds, start=1):
        seed = int(seed)
        print("\n" + "=" * 70)
        print(f"{dataset_name} - {synth_name} - run {run_id}/{n_runs}")
        print(f"Seed: {seed}")
        print("=" * 70)

        if retrain_each_run:
            train_start_time = time.time()
            synthesizer = train_fn(df, metadata, seed=seed, **synth_params)
            training_time = time.time() - train_start_time
            parameters = synthesizer.get_parameters()
        else:
            # Only the internal RNG state (advanced by np.random.seed) affects
            # sampling; fitting was already performed once above.
            np.random.seed(seed)
            synthesizer = fitted_synthesizer
            training_time = base_training_time
            parameters = base_parameters

        # ---------------- Sampling ----------------
        sample_start_time = time.time()
        synthetic_df = synthesizer.sample(num_rows=len(df))
        sampling_time = time.time() - sample_start_time

        # ---------------- Save synthetic dataset ----------------
        synthetic_path = os.path.join(
            output_dir, f"{dataset_name}_{synth_name}_{run_id:02d}.csv"
        )
        synthetic_df.to_csv(synthetic_path, index=False)

        dataset_hash = _hash_dataframe(synthetic_df)

        run_results.append(
            {
                "run": run_id,
                "seed": seed,
                "training_time_sec": training_time,
                "sampling_time_sec": sampling_time,
                "dataset_hash": dataset_hash,
            }
        )

        parameter_record = {"run": run_id, "seed": seed}
        parameter_record.update(parameters)
        parameter_results.append(parameter_record)

    total_experiment_time = time.time() - experiment_start_time

    run_summary_df = pd.DataFrame(run_results)
    parameters_df = pd.DataFrame(parameter_results)

    _print_experiment_report(
        dataset_name, synth_name, run_summary_df, total_experiment_time, n_runs
    )

    return run_summary_df, parameters_df


def _print_experiment_report(
    dataset_name: str,
    synth_name: str,
    run_summary_df: pd.DataFrame,
    total_experiment_time: float,
    n_runs: int,
) -> None:
    """Print a short, publication-friendly summary of one experiment."""
    print("\n" + "=" * 70)
    print(f"{dataset_name} {synth_name.upper()} EXPERIMENT COMPLETED")
    print("=" * 70)
    print(f"Number of runs : {n_runs}")

    mean_training_time = run_summary_df["training_time_sec"].mean()
    std_training_time = run_summary_df["training_time_sec"].std()
    print("\nTraining time")
    print(f"Mean : {mean_training_time:.4f} sec")
    print(f"Std  : {std_training_time:.4f} sec")

    mean_sampling_time = run_summary_df["sampling_time_sec"].mean()
    std_sampling_time = run_summary_df["sampling_time_sec"].std()
    print("\nSampling time")
    print(f"Mean : {mean_sampling_time:.4f} sec")
    print(f"Std  : {std_sampling_time:.4f} sec")

    print(f"\nTotal experiment time : {total_experiment_time:.4f} sec")

    n_unique_datasets = run_summary_df["dataset_hash"].nunique()
    print(f"Unique datasets generated : {n_unique_datasets}/{n_runs}")


def save_experiment_results(
    output_dir: str,
    dataset_name: str,
    synth_name: str,
    run_summary_df: pd.DataFrame,
    parameters_df: pd.DataFrame,
    environment_df: pd.DataFrame,
) -> None:
    """Save run summary, parameter summary, and environment info as CSV files."""
    run_summary_df.to_csv(
        os.path.join(output_dir, f"{dataset_name}_{synth_name}_run_summary.csv"),
        index=False,
    )
    parameters_df.to_csv(
        os.path.join(
            output_dir, f"{dataset_name}_{synth_name}_parameters_all_runs.csv"
        ),
        index=False,
    )
    environment_df.to_csv(
        os.path.join(output_dir, f"{dataset_name}_{synth_name}_environment.csv"),
        index=False,
    )


# ==============================================================================
# High-level pipeline: one dataset, all configured synthesizers
# ==============================================================================

def run_dataset_pipeline(
    dataset_name: str,
    data_path: str,
    output_base_dir: str,
    synth_configs: Dict[str, Dict[str, Any]],
    n_runs: int = 50,
    base_seed: int = 0,
) -> Dict[str, Tuple[pd.DataFrame, pd.DataFrame]]:
    """
    Run the full synthetic-data generation procedure for a single dataset,
    across every synthesizer listed in `synth_configs`.

    Parameters
    ----------
    dataset_name : str
        Short dataset identifier (e.g. "PIMA", "AIDS", "REIN"). Used to build
        output filenames and per-synthesizer output subfolders.
    data_path : str
        Path to the dataset's training CSV file.
    output_base_dir : str
        Base directory under which per-synthesizer subfolders are created,
        e.g. "<output_base_dir>/<dataset_name>_synthetic_datasets_CTGAN".
    synth_configs : dict
        Mapping {synthesizer_name: hyperparameters_dict}. Keys must be among
        SYNTHESIZER_TRAINERS ("GaussianCopula", "CTGAN", "TVAE"). Only the
        synthesizers present in this dict are run, which lets each dataset
        use its own subset/parameterisation of generators.
    n_runs : int
        Number of independent generation runs per synthesizer.
    base_seed : int
        Master seed used to derive the `n_runs` per-run seeds.

    Returns
    -------
    dict
        Mapping {synthesizer_name: (run_summary_df, parameters_df)}.
    """
    print("\n" + "#" * 70)
    print(f"# DATASET: {dataset_name}")
    print("#" * 70)

    seeds = generate_seeds(n_runs=n_runs, base_seed=base_seed)
    df, metadata = load_dataset(data_path, table_name=dataset_name)
    environment_df = get_environment_info()

    results: Dict[str, Tuple[pd.DataFrame, pd.DataFrame]] = {}

    for synth_name, synth_params in synth_configs.items():
        if synth_name not in SYNTHESIZER_TRAINERS:
            raise ValueError(
                f"Unknown synthesizer '{synth_name}'. "
                f"Available options: {list(SYNTHESIZER_TRAINERS)}"
            )

        output_dir = os.path.join(
            output_base_dir, f"{dataset_name}_synthetic_datasets_{synth_name}"
        )

        run_summary_df, parameters_df = run_synthesizer_experiment(
            df=df,
            metadata=metadata,
            dataset_name=dataset_name,
            synth_name=synth_name,
            synth_params=synth_params,
            seeds=seeds,
            output_dir=output_dir,
        )

        save_experiment_results(
            output_dir=output_dir,
            dataset_name=dataset_name,
            synth_name=synth_name,
            run_summary_df=run_summary_df,
            parameters_df=parameters_df,
            environment_df=environment_df,
        )

        results[synth_name] = (run_summary_df, parameters_df)

    return results


# ==============================================================================
# Master driver: run the procedure for several datasets in one call
# ==============================================================================

def run_all_datasets(
    dataset_configs: Dict[str, Dict[str, Any]],
    n_runs: int = 50,
    base_seed: int = 0,
) -> Dict[str, Dict[str, Tuple[pd.DataFrame, pd.DataFrame]]]:
    """
    Run the full pipeline for every dataset listed in `dataset_configs`.

    This is the single entry point needed to reproduce the whole study
    (PIMA, AIDS, REIN, ...): each dataset can define its own data path,
    output directory, and generator hyperparameters, while the underlying
    procedure (seeding, training, sampling, saving, reporting) is shared.

    Parameters
    ----------
    dataset_configs : dict
        Mapping {dataset_name: kwargs_for_run_dataset_pipeline}, where each
        value is a dict with keys "data_path", "output_base_dir", and
        "synth_configs" (see `run_dataset_pipeline`).
    n_runs : int
        Default number of runs per synthesizer, used unless overridden in a
        given dataset's config (key "n_runs").
    base_seed : int
        Default master seed, used unless overridden in a given dataset's
        config (key "base_seed").

    Returns
    -------
    dict
        Mapping {dataset_name: {synth_name: (run_summary_df, parameters_df)}}.
    """
    all_results: Dict[str, Dict[str, Tuple[pd.DataFrame, pd.DataFrame]]] = {}

    for dataset_name, cfg in dataset_configs.items():
        all_results[dataset_name] = run_dataset_pipeline(
            dataset_name=dataset_name,
            data_path=cfg["data_path"],
            output_base_dir=cfg["output_base_dir"],
            synth_configs=cfg["synth_configs"],
            n_runs=cfg.get("n_runs", n_runs),
            base_seed=cfg.get("base_seed", base_seed),
        )

    return all_results


### Train datasets

In [7]:
# ==============================================================================
# Example configuration
# ==============================================================================
#
# Edit the paths and hyperparameters below to match your project layout.
# Each dataset can have its own CTGAN/TVAE hyperparameters, and can even
# include only a subset of the three synthesizers if desired.

DATA_ROOT = "Bureau/CHIMERA_SYNTHETIC_HEALTH_DATA_GENERATION_V3/data"

DATASET_CONFIGS: Dict[str, Dict[str, Any]] = {
    "PIMA": {
        "data_path": os.path.join(DATA_ROOT, "PIMA_train.csv"),
        "output_base_dir": DATA_ROOT,
        "n_runs": 50,
        "base_seed": 0,
        "synth_configs": {
            "GaussianCopula": {},
            "CTGAN": {
                "epochs": 600,
                "batch_size": 70,
                "generator_dim": (256, 256),
                "discriminator_dim": (256, 256),
                "cuda": False,
            },
            "TVAE": {
                "epochs": 600,
                "batch_size": 500,
                "embedding_dim": 128,
                "compress_dims": (128, 128),
                "decompress_dims": (128, 128),
                "l2scale": 1e-5,
                "cuda": False,
            },
        },
    },
    "AIDS": {
        "data_path": os.path.join(DATA_ROOT, "AIDS_train.csv"),
        "output_base_dir": DATA_ROOT,
        "n_runs": 50,
        "base_seed": 0,
        "synth_configs": {
            "GaussianCopula": {},
            # TODO: set AIDS-specific CTGAN/TVAE hyperparameters.
            "CTGAN": {
                "epochs": 600,
                "batch_size": 200,
                "generator_dim": (128, 256),
                "discriminator_dim": (256, 128),
                "cuda": False,
            },
            "TVAE": {
                "epochs": 3000,
                "batch_size": 256,
                "embedding_dim": 128,
                "compress_dims": (128, 256),
                "decompress_dims": (128, 256),
                "l2scale": 1e-3,
                "cuda": False,
            },
        },
    },
    "REIN": {
        "data_path": os.path.join(DATA_ROOT, "REIN_train.csv"),
        "output_base_dir": DATA_ROOT,
        "n_runs": 50,
        "base_seed": 0,
        "synth_configs": {
            "GaussianCopula": {},
            # TODO: set REIN-specific CTGAN/TVAE hyperparameters.
            "CTGAN": {
                "epochs": 500,
                "batch_size": 3000,
                "generator_dim": (128, 128),
                "discriminator_dim": (256, 256),
                "cuda": False,
            },
            "TVAE": {
                "epochs": 2000,
                "batch_size": 500,
                "embedding_dim": 64,
                "compress_dims": (256, 128),
                "decompress_dims": (128, 256),
                "l2scale": 1e-5,
                "cuda": False,
            },
        },
    },
}


if __name__ == "__main__":
    # Running this file directly (or executing this cell in Jupyter)
    # reproduces the full study for every dataset in DATASET_CONFIGS.
    results = run_all_datasets(DATASET_CONFIGS)


######################################################################
# DATASET: PIMA
######################################################################

Dataset information
-------------------
Dataset name : PIMA
Rows         : 537
Columns      : 9
DETERMINISTIC MODE enabled (seed=1826701614)


/home/niang/.local/lib/python3.10/site-packages/sdv/single_table/base.py:105: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(



PIMA - GaussianCopula - run 1/50
Seed: 1826701614

PIMA - GaussianCopula - run 2/50
Seed: 1367864806

PIMA - GaussianCopula - run 3/50
Seed: 1097657231

PIMA - GaussianCopula - run 4/50
Seed: 579362555

PIMA - GaussianCopula - run 5/50
Seed: 661058651

PIMA - GaussianCopula - run 6/50
Seed: 87989972

PIMA - GaussianCopula - run 7/50
Seed: 161576974

PIMA - GaussianCopula - run 8/50
Seed: 35492826

PIMA - GaussianCopula - run 9/50
Seed: 376383645

PIMA - GaussianCopula - run 10/50
Seed: 1746484539

PIMA - GaussianCopula - run 11/50
Seed: 1394609702

PIMA - GaussianCopula - run 12/50
Seed: 1960127675

PIMA - GaussianCopula - run 13/50
Seed: 1081530686

PIMA - GaussianCopula - run 14/50
Seed: 1302740407

PIMA - GaussianCopula - run 15/50
Seed: 2084654369

PIMA - GaussianCopula - run 16/50
Seed: 1566581934

PIMA - GaussianCopula - run 17/50
Seed: 1357791198

PIMA - GaussianCopula - run 18/50
Seed: 1167425778

PIMA - GaussianCopula - run 19/50
Seed: 1202413557

PIMA - GaussianCopula - run 

Gen. (-2.21) | Discrim. (-0.26): 100%|████████| 600/600 [01:01<00:00,  9.74it/s]



PIMA - CTGAN - run 2/50
Seed: 1367864806
DETERMINISTIC MODE enabled (seed=1367864806)


...


REIN - TVAE - run 49/50
Seed: 552547096
DETERMINISTIC MODE enabled (seed=552547096)


Loss: -1.865: 100%|█████████████████████████| 2000/2000 [40:07<00:00,  1.20s/it]



REIN - TVAE - run 50/50
Seed: 1321529463
DETERMINISTIC MODE enabled (seed=1321529463)


Loss: -0.626: 100%|█████████████████████████| 2000/2000 [38:50<00:00,  1.17s/it]



REIN TVAE EXPERIMENT COMPLETED
Number of runs : 50

Training time
Mean : 2528.0606 sec
Std  : 205.9208 sec

Sampling time
Mean : 0.2902 sec
Std  : 0.0268 sec

Total experiment time : 126424.8335 sec
Unique datasets generated : 50/50


### Houldout set datasets

In [14]:
# ==============================================================================
# High-level pipeline: one dataset, all configured synthesizers
# ==============================================================================

def run_dataset_pipeline(
    dataset_name: str,
    data_path: str,
    output_base_dir: str,
    synth_configs: Dict[str, Dict[str, Any]],
    n_runs: int = 50,
    base_seed: int = 0,
) -> Dict[str, Tuple[pd.DataFrame, pd.DataFrame]]:
    """
    Run the full synthetic-data generation procedure for a single dataset,
    across every synthesizer listed in `synth_configs`.

    Parameters
    ----------
    dataset_name : str
        Short dataset identifier (e.g. "PIMA", "AIDS", "REIN"). Used to build
        output filenames and per-synthesizer output subfolders.
    data_path : str
        Path to the dataset's training CSV file.
    output_base_dir : str
        Base directory under which per-synthesizer subfolders are created,
        e.g. "<output_base_dir>/<dataset_name>_synthetic_datasets_CTGAN".
    synth_configs : dict
        Mapping {synthesizer_name: hyperparameters_dict}. Keys must be among
        SYNTHESIZER_TRAINERS ("GaussianCopula", "CTGAN", "TVAE"). Only the
        synthesizers present in this dict are run, which lets each dataset
        use its own subset/parameterisation of generators.
    n_runs : int
        Number of independent generation runs per synthesizer.
    base_seed : int
        Master seed used to derive the `n_runs` per-run seeds.

    Returns
    -------
    dict
        Mapping {synthesizer_name: (run_summary_df, parameters_df)}.
    """
    print("\n" + "#" * 70)
    print(f"# DATASET: {dataset_name}")
    print("#" * 70)

    seeds = generate_seeds(n_runs=n_runs, base_seed=base_seed)
    df, metadata = load_dataset(data_path, table_name=dataset_name)
    environment_df = get_environment_info()

    results: Dict[str, Tuple[pd.DataFrame, pd.DataFrame]] = {}

    for synth_name, synth_params in synth_configs.items():
        if synth_name not in SYNTHESIZER_TRAINERS:
            raise ValueError(
                f"Unknown synthesizer '{synth_name}'. "
                f"Available options: {list(SYNTHESIZER_TRAINERS)}"
            )

        output_dir = os.path.join(
            output_base_dir, f"{dataset_name}_synthetic_datasets_{synth_name}_trn"
        )

        run_summary_df, parameters_df = run_synthesizer_experiment(
            df=df,
            metadata=metadata,
            dataset_name=dataset_name,
            synth_name=synth_name,
            synth_params=synth_params,
            seeds=seeds,
            output_dir=output_dir,
        )

        save_experiment_results(
            output_dir=output_dir,
            dataset_name=dataset_name,
            synth_name=synth_name,
            run_summary_df=run_summary_df,
            parameters_df=parameters_df,
            environment_df=environment_df,
        )

        results[synth_name] = (run_summary_df, parameters_df)

    return results


In [ ]:
# ==============================================================================
# Example configuration
# ==============================================================================
#
# Edit the paths and hyperparameters below to match your project layout.
# Each dataset can have its own CTGAN/TVAE hyperparameters, and can even
# include only a subset of the three synthesizers if desired.

DATA_ROOT = "Bureau/CHIMERA_SYNTHETIC_HEALTH_DATA_GENERATION_V3/data"

DATASET_CONFIGS: Dict[str, Dict[str, Any]] = {
    "PIMA": {
        "data_path": os.path.join(DATA_ROOT, "PIMA_trn.csv"),
        "output_base_dir": DATA_ROOT,
        "n_runs": 50,
        "base_seed": 0,
        "synth_configs": {
            "GaussianCopula": {},
            "CTGAN": {
                "epochs": 600,
                "batch_size": 70,
                "generator_dim": (256, 256),
                "discriminator_dim": (256, 256),
                "cuda": False,
            },
            "TVAE": {
                "epochs": 600,
                "batch_size": 500,
                "embedding_dim": 128,
                "compress_dims": (128, 128),
                "decompress_dims": (128, 128),
                "l2scale": 1e-5,
                "cuda": False,
            },
        },
    },
    "AIDS": {
        "data_path": os.path.join(DATA_ROOT, "AIDS_trn.csv"),
        "output_base_dir": DATA_ROOT,
        "n_runs": 50,
        "base_seed": 0,
        "synth_configs": {
            "GaussianCopula": {},
            # TODO: set AIDS-specific CTGAN/TVAE hyperparameters.
            "CTGAN": {
                "epochs": 600,
                "batch_size": 200,
                "generator_dim": (128, 256),
                "discriminator_dim": (256, 128),
                "cuda": False,
            },
            "TVAE": {
                "epochs": 3000,
                "batch_size": 256,
                "embedding_dim": 128,
                "compress_dims": (128, 256),
                "decompress_dims": (128, 256),
                "l2scale": 1e-3,
                "cuda": False,
            },
        },
    },
    "REIN": {
        "data_path": os.path.join(DATA_ROOT, "REIN_trn.csv"),
        "output_base_dir": DATA_ROOT,
        "n_runs": 50,
        "base_seed": 0,
        "synth_configs": {
            "GaussianCopula": {},
            # TODO: set REIN-specific CTGAN/TVAE hyperparameters.
            "CTGAN": {
                "epochs": 500,
                "batch_size": 3000,
                "generator_dim": (128, 128),
                "discriminator_dim": (256, 256),
                "cuda": False,
            },
            "TVAE": {
                "epochs": 2000,
                "batch_size": 500,
                "embedding_dim": 64,
                "compress_dims": (256, 128),
                "decompress_dims": (128, 256),
                "l2scale": 1e-5,
                "cuda": False,
            },
        },
    },
}


if __name__ == "__main__":
    # Running this file directly (or executing this cell in Jupyter)
    # reproduces the full study for every dataset in DATASET_CONFIGS.
    results = run_all_datasets(DATASET_CONFIGS)